# SC 612 104 Essential Data Science
## คาบที่ 11: Data Visualization — matplotlib, seaborn, และ Visualization for Insight

**ผู้สอน:** อ.ดร.พิชญา วิรัชโชติเสถียร (Pitchaya Wiratchotisatian)
**ภาควิชาสถิติ คณะวิทยาศาสตร์ มหาวิทยาลัยขอนแก่น**

---

### เนื้อหาในคาบนี้

**ส่วนที่ 1: matplotlib พื้นฐาน**
1. Line Chart, Scatter Plot, Bar Chart, Histogram
2. Title, Label, Legend, Style

**ส่วนที่ 2: เครื่องมือขั้นสูงกว่า**\
3. Subplot — วางกราฟหลายอันในภาพเดียว\
4. `pandas.plot()` — plot ตรงจาก DataFrame\
5. seaborn: Boxplot, Heatmap (Correlation), Pairplot

**ส่วนที่ 3: Visualization for Insight**\
6. เลือก Chart Type ให้เหมาะกับข้อมูล\
7. ตั้งชื่อกราฟให้สื่อความหมาย\
8. Storytelling เบื้องต้นด้วยข้อมูล

**แบบฝึกหัดท้ายคาบ** 

> 📁 **ไฟล์ที่ต้องใช้ในคาบนี้:** `world_weather_sample.csv`, `city_info.csv`, `region_info.csv` 


In [ ]:
# ถ้าใช้ Google Colab และยังไม่ได้อัปโหลดไฟล์ ให้รันเซลล์นี้เพื่ออัปโหลดทั้ง 3 ไฟล์พร้อมกัน
try:
    from google.colab import files
    uploaded = files.upload()   # เลือก world_weather_sample.csv, city_info.csv, region_info.csv
except ImportError:
    print("ไม่ได้รันบน Colab - ข้ามขั้นตอนนี้ (ตรวจสอบว่าไฟล์ CSV ทั้ง 3 อยู่ในโฟลเดอร์เดียวกับ Notebook นี้)")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ตั้งค่าฟอนต์ให้รองรับภาษาไทยในกราฟ (ถ้าไม่มีฟอนต์นี้ในเครื่อง จะใช้ฟอนต์ default แทน)
import matplotlib.font_manager as fm
thai_fonts = ["Loma", "TH Sarabun New", "Tahoma", "Noto Sans Thai"]
available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in thai_fonts:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False   # ป้องกัน warning เรื่องเครื่องหมายลบหายไปจากบางฟอนต์

print("ติดตั้งไลบรารีพร้อมใช้งาน")

---
## 0. เตรียมข้อมูล: Merge + Clean 

ก่อนสร้างกราฟ เราจะ **merge 3 ตาราง** เข้าด้วยกัน แล้ว **ทำความสะอาด** เพื่อให้กราฟที่ได้สื่อความหมายจริง ไม่ถูกบิดเบือนจากข้อมูลสกปรก


In [ ]:
# โหลดข้อมูลทั้ง 3 ไฟล์
df = pd.read_csv("world_weather_sample.csv")
df["date"] = pd.to_datetime(df["date"])
city_info = pd.read_csv("city_info.csv")
region_info = pd.read_csv("region_info.csv")

# Merge 3 ตารางเข้าด้วยกัน (เทคนิคจาก Session 9)
data = (
    df
    .merge(city_info[["city", "population_millions", "timezone"]], on="city", how="left")
    .merge(region_info, on="region", how="left")
)
print("ขนาดข้อมูลหลัง merge:", data.shape)
data.head()

In [ ]:
# ทำความสะอาดข้อมูล 

# 1. ลบแถวที่ temperature_c ผิดปกติอย่างรุนแรง (นอกช่วงที่เป็นไปได้ทางฟิสิกส์)
data = data[(data["temperature_c"].isnull()) | (data["temperature_c"].between(-60, 60))]

# 2. Cap ค่า humidity_pct ให้อยู่ในช่วง 0-100 และ wind_speed_kmh ให้ไม่ติดลบ
data["humidity_pct"] = data["humidity_pct"].clip(lower=0, upper=100)
data["wind_speed_kmh"] = data["wind_speed_kmh"].clip(lower=0)

# 3. Fill missing values ที่เหลือด้วย median (ทนทานต่อ outliers ที่เหลือ)
numeric_cols = ["temperature_c", "humidity_pct", "rainfall_mm", "wind_speed_kmh", "pressure_hpa"]
for col in numeric_cols:
    data[col] = data[col].fillna(data[col].median())

# 4. ลบแถวซ้ำ 
data = data.drop_duplicates()

# 5. ทำให้ตัวพิมพ์ของ country สม่ำเสมอ
data["country"] = data["country"].str.title()

print("ขนาดข้อมูลหลังทำความสะอาด:", data.shape)
print("\nMissing values ที่เหลือ:")
print(data[numeric_cols].isnull().sum())
data.describe()[numeric_cols].round(1)

**ตอนนี้ `data` คือ DataFrame ที่สะอาดและพร้อมใช้สร้างกราฟแล้ว** — จะใช้ตัวแปรนี้ตลอดทั้ง Notebook


---
# ส่วนที่ 1: matplotlib พื้นฐาน

## ทำไมต้องมี Data Visualization?

`.describe()`, `.groupby()` ที่เรียนมาให้ "ตัวเลข" แต่สมองมนุษย์มองเห็น**รูปแบบ (pattern)** ได้ดีกว่าจากภาพมากกว่าตัวเลขล้วนๆ — กราฟที่ดีช่วยให้:
- เห็นแนวโน้ม ความสัมพันธ์ และความผิดปกติได้เร็วกว่าอ่านตาราง
- สื่อสารผลการวิเคราะห์ให้คนอื่นเข้าใจได้ง่าย (ไม่ใช่ทุกคนถนัดอ่านตัวเลข)
- ตรวจสอบสมมติฐานเบื้องต้นก่อนทำการวิเคราะห์เชิงลึก

**matplotlib** คือไลบรารีพื้นฐานที่สุดสำหรับสร้างกราฟด้วย Python — ไลบรารีอื่น (รวมถึง seaborn และ `pandas.plot()`) ต่างก็สร้างอยู่บน matplotlib ทั้งสิ้น


---
## 1. Line Chart — กราฟเส้น

ใช้แสดง**แนวโน้มตามเวลา** หรือลำดับที่มีความต่อเนื่อง — เหมาะกับข้อมูล time series ที่สุด


In [ ]:
bangkok = data[data["city"] == "Bangkok"].sort_values("date")

plt.figure(figsize=(10, 5))
plt.plot(bangkok["date"], bangkok["temperature_c"])
plt.show()

**กราฟนี้ "ใช้งานได้" แต่ยังไม่ดี** — ไม่มีชื่อกราฟ ไม่มีชื่อแกน คนอื่นดูไม่รู้ว่ากราฟนี้คืออะไร หัวข้อถัดไป (Title/Label/Legend/Style) จะแก้ปัญหานี้


---
## 2. Scatter Plot — กราฟจุดกระจาย

ใช้แสดง**ความสัมพันธ์ระหว่าง 2 ตัวแปรตัวเลข** — แต่ละจุดคือ 1 record ของข้อมูล


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(data["humidity_pct"], data["rainfall_mm"], alpha=0.3)
plt.show()

**🔍 สังเกต:** ลองดูรูปแบบการกระจายตัวของจุด — ความชื้นสูงกับปริมาณฝนมีความสัมพันธ์กันไหม? (alpha=0.3 ทำให้จุดโปร่งแสง เห็นความหนาแน่นของจุดที่ซ้อนกันได้ดีขึ้นเมื่อมีข้อมูลจำนวนมาก)


---
## 3. Bar Chart — กราฟแท่ง

ใช้เปรียบเทียบ**ค่าระหว่างกลุ่ม** (categorical) — เชื่อมกับ `groupby()` ที่เรียนมาจาก Session 9


In [ ]:
avg_temp_by_region = data.groupby("region")["temperature_c"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(avg_temp_by_region.index, avg_temp_by_region.values)
plt.show()

### 3.1 Horizontal Bar Chart — เหมาะเมื่อชื่อกลุ่มยาว (อ่านง่ายกว่า)


In [ ]:
avg_temp_by_city = data.groupby("city")["temperature_c"].mean().sort_values()

plt.figure(figsize=(8, 8))
plt.barh(avg_temp_by_city.index, avg_temp_by_city.values)
plt.show()

---
## 4. Histogram — ฮิสโตแกรม

ใช้แสดง**การกระจายตัว (distribution)** ของข้อมูลตัวเลข 1 คอลัมน์ — ต่างจาก bar chart ที่เทียบระหว่างกลุ่ม ฮิสโตแกรมแบ่ง "ช่วงค่า" (bins) แล้วนับว่ามีกี่ค่าตกอยู่ในแต่ละช่วง


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(data["temperature_c"], bins=30)
plt.show()

**อ่านฮิสโตแกรมอย่างไร:** แกน x คือช่วงค่าของอุณหภูมิ แกน y คือ "จำนวนวัน/record" ที่อุณหภูมิตกอยู่ในช่วงนั้น — รูปทรงของฮิสโตแกรมบอกการกระจายตัว เช่น โค้งระฆัง (normal distribution), เบ้ซ้าย/ขวา (skewed), หรือมี 2 ยอด (bimodal — อาจบอกใบ้ว่าข้อมูลมี 2 กลุ่มย่อยซ่อนอยู่)

### 4.1 ปรับจำนวน `bins` ส่งผลต่อการตีความอย่างไร


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, n_bins in zip(axes, [10, 30, 100]):
    ax.hist(data["temperature_c"], bins=n_bins)
    ax.set_title(f"bins={n_bins}")
plt.tight_layout()
plt.show()

**🔍 สังเกต:** `bins` น้อยเกินไป → เห็นภาพรวมหยาบเกินไป (อาจพลาดรายละเอียดสำคัญ) ส่วน `bins` มากเกินไป → เห็นแต่ noise จนมองไม่เห็นรูปแบบโดยรวม — ไม่มีค่าที่ "ถูกต้องที่สุด" ตายตัว ต้องลองปรับดูตามข้อมูลจริง


---
## 5. Title, Label, Legend, Style — ทำให้กราฟ "สื่อความหมาย"

กราฟที่ผ่านมาทำงานถูกต้อง แต่**ไม่มีใครเข้าใจ**ถ้าไม่มีบริบท — องค์ประกอบ 4 อย่างนี้คือสิ่งที่กราฟทุกอันที่จะนำไปใช้จริง (รายงาน, นำเสนอ) ต้องมีเสมอ

| องค์ประกอบ | คำสั่ง | หน้าที่ |
|---|---|---|
| **Title** | `plt.title()` | บอกว่ากราฟนี้คืออะไร |
| **Label** | `plt.xlabel()`, `plt.ylabel()` | บอกว่าแต่ละแกนคือตัวแปรอะไร หน่วยอะไร |
| **Legend** | `plt.legend()` | บอกว่าแต่ละเส้น/สี/จุด แทนอะไร (เมื่อมีหลายชุดข้อมูลในกราฟเดียว) |
| **Style** | `color`, `linestyle`, `marker`, ฯลฯ | ช่วยแยกแยะและทำให้กราฟอ่านง่ายขึ้น |

### 5.1 ตัวอย่าง Line Chart ที่มีองค์ประกอบครบ


In [ ]:
bangkok = data[data["city"] == "Bangkok"].sort_values("date")

plt.figure(figsize=(10, 5))
plt.plot(bangkok["date"], bangkok["temperature_c"], color="firebrick", linewidth=1.5)
plt.title("อุณหภูมิรายวันที่กรุงเทพฯ (มกราคม - มีนาคม 2025)")
plt.xlabel("วันที่")
plt.ylabel("อุณหภูมิ (°C)")
plt.xticks(rotation=30)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**เทียบกับกราฟแรกที่ไม่มีองค์ประกอบเหล่านี้** — กราฟนี้ดูครั้งแรกก็เข้าใจได้ทันทีว่า "นี่คือกราฟอุณหภูมิรายวันของกรุงเทพฯ" โดยไม่ต้องถามใคร

### 5.2 Legend — เมื่อมีหลายชุดข้อมูลในกราฟเดียว

ใส่ `label=` ตอนวาดกราฟแต่ละเส้น แล้วเรียก `plt.legend()` ตอนท้าย


In [ ]:
cities_to_compare = ["Bangkok", "Tokyo", "London"]
colors = ["firebrick", "steelblue", "seagreen"]

plt.figure(figsize=(10, 5))
for city, color in zip(cities_to_compare, colors):
    city_data = data[data["city"] == city].sort_values("date")
    plt.plot(city_data["date"], city_data["temperature_c"], label=city, color=color)

plt.title("เปรียบเทียบอุณหภูมิรายวัน: Bangkok vs Tokyo vs London")
plt.xlabel("วันที่")
plt.ylabel("อุณหภูมิ (°C)")
plt.legend(title="เมือง")
plt.xticks(rotation=30)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Style — ตัวเลือกการตกแต่งที่ใช้บ่อย

| พารามิเตอร์ | ใช้กับกราฟ | ตัวอย่างค่า |
|---|---|---|
| `color` | ทุกกราฟ | `"red"`, `"#1f77b4"`, `"steelblue"` |
| `linestyle` | line chart | `"-"` (ทึบ), `"--"` (ประ), `":"` (จุด) |
| `marker` | line/scatter | `"o"` (กลม), `"s"` (สี่เหลี่ยม), `"^"` (สามเหลี่ยม) |
| `alpha` | ทุกกราฟ | 0 (โปร่งใสหมด) ถึง 1 (ทึบหมด) |
| `figsize` | ตอนสร้าง figure | `(กว้าง, สูง)` หน่วยนิ้ว |


In [ ]:
plt.figure(figsize=(10, 5))

tokyo = data[data["city"] == "Tokyo"].sort_values("date")
plt.plot(tokyo["date"], tokyo["temperature_c"],
         color="darkorange", linestyle="--", marker="o", markersize=3, alpha=0.7)

plt.title("อุณหภูมิรายวันที่ Tokyo (พร้อม marker แต่ละจุดข้อมูล)")
plt.xlabel("วันที่")
plt.ylabel("อุณหภูมิ (°C)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 5.4 ใช้ Style Sheet สำเร็จรูปของ matplotlib

matplotlib มี theme สำเร็จรูปให้เลือกใช้ทั้งหน้า ไม่ต้องตั้งค่าทีละกราฟ


In [ ]:
print("Style ที่มีให้เลือก (ตัวอย่างบางส่วน):", plt.style.available[:8])

current_font = plt.rcParams["font.family"]   # จำฟอนต์ปัจจุบันไว้ก่อน เพราะ style ใหม่อาจทับฟอนต์ไทยที่ตั้งไว้

with plt.style.context("seaborn-v0_8-whitegrid"):
    plt.rcParams["font.family"] = current_font   # ตั้งฟอนต์ไทยกลับคืนหลังเข้า style context
    plt.figure(figsize=(10, 5))
    plt.plot(bangkok["date"], bangkok["temperature_c"], color="firebrick")
    plt.title("อุณหภูมิรายวันที่กรุงเทพฯ (ใช้ style seaborn-v0_8-whitegrid)")
    plt.xlabel("วันที่")
    plt.ylabel("อุณหภูมิ (°C)")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

> 💡 **เกร็ดเสริม:** ใช้ `with plt.style.context("ชื่อสไตล์"):` เพื่อใช้สไตล์นั้นแค่ในบล็อกโค้ดนี้ (ไม่กระทบกราฟอื่นที่จะวาดต่อไป) ถ้าอยากใช้ทั้ง Notebook ให้เรียก `plt.style.use("ชื่อสไตล์")` ครั้งเดียวตอนต้นไฟล์


---
# ส่วนที่ 2: เครื่องมือขั้นสูงกว่า

## 6. Subplot — วางกราฟหลายอันในภาพเดียว

`plt.subplots(แถว, คอลัมน์)` สร้าง "กริด" ของกราฟย่อยหลายอันในภาพเดียว — มีประโยชน์มากเวลาต้องการเปรียบเทียบหลายมุมมองพร้อมกัน


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# axes เป็น array 2 มิติ (เชื่อมกับ NumPy ที่เรียนมา) เข้าถึงแต่ละช่องด้วย [แถว, คอลัมน์]
axes[0, 0].plot(bangkok["date"], bangkok["temperature_c"], color="firebrick")
axes[0, 0].set_title("อุณหภูมิรายวัน (Bangkok)")

axes[0, 1].hist(data["temperature_c"], bins=30, color="steelblue")
axes[0, 1].set_title("การกระจายตัวของอุณหภูมิ (ทุกเมือง)")

avg_temp_by_region = data.groupby("region")["temperature_c"].mean().sort_values(ascending=False)
axes[1, 0].bar(avg_temp_by_region.index, avg_temp_by_region.values, color="seagreen")
axes[1, 0].set_title("อุณหภูมิเฉลี่ยแยกตาม Region")
axes[1, 0].tick_params(axis="x", rotation=30)

axes[1, 1].scatter(data["humidity_pct"], data["rainfall_mm"], alpha=0.3, color="purple")
axes[1, 1].set_title("ความชื้น vs ปริมาณฝน")

plt.tight_layout()
plt.show()

**สังเกต:** เมื่อใช้ `plt.subplots()` ต้องเรียก `.set_title()`, `.set_xlabel()` แทน `plt.title()`, `plt.xlabel()` ปกติ (เพราะ `ax` แต่ละช่องเป็น object ของตัวเอง — เชื่อมกับ OOP/method ที่เรียนมา: `axes[0,0]` คือ object หนึ่ง มี method ของตัวเอง)

### 6.1 Subplot แบบแถวเดียว/คอลัมน์เดียว (ง่ายกว่าเวลาไม่ต้องการกริด 2 มิติ)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cities_subset = ["Bangkok", "Tokyo", "London"]
for ax, city in zip(axes, cities_subset):
    city_data = data[data["city"] == city].sort_values("date")
    ax.plot(city_data["date"], city_data["temperature_c"])
    ax.set_title(city)
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("เปรียบเทียบอุณหภูมิรายวัน 3 เมือง", y=1.02)
plt.tight_layout()
plt.show()

> 💡 **เกร็ดเสริม:** `plt.suptitle()` ใส่ชื่อรวมให้ทั้งภาพ (เหนือกราฟย่อยทั้งหมด) ต่างจาก `ax.set_title()` ที่ใส่ชื่อให้กราฟย่อยแต่ละอัน


---
## 7. `pandas.plot()` — Plot โดยตรงจาก DataFrame

pandas มีวิธี `.plot()` ติดตัวมาเลย ไม่ต้องเรียก `plt.plot()` แยก — สร้างกราฟเร็วกว่าตอนสำรวจข้อมูล (Exploratory Data Analysis) เพราะเขียนโค้ดน้อยกว่า

**ข้อสำคัญ:** `.plot()` ของ pandas สร้างอยู่บน matplotlib เบื้องหลัง ดังนั้นยังใช้ `plt.title()`, `plt.xlabel()` ฯลฯ ต่อท้ายได้ตามปกติ


In [ ]:
bangkok = data[data["city"] == "Bangkok"].sort_values("date")

bangkok.plot(x="date", y="temperature_c", figsize=(10, 5), color="firebrick")
plt.title("อุณหภูมิรายวันที่กรุงเทพฯ (วาดด้วย pandas.plot())")
plt.ylabel("อุณหภูมิ (°C)")
plt.tight_layout()
plt.show()

### 7.1 `pandas.plot()` กับ `kind=` — เลือกชนิดกราฟ

พารามิเตอร์ `kind` กำหนดชนิดกราฟ: `"line"` (default), `"bar"`, `"barh"`, `"hist"`, `"scatter"`, `"box"`


In [ ]:
avg_temp_by_region = data.groupby("region")["temperature_c"].mean().sort_values(ascending=False)

avg_temp_by_region.plot(kind="bar", figsize=(10, 5), color="seagreen")
plt.title("อุณหภูมิเฉลี่ยแยกตาม Region (วาดด้วย pandas.plot(kind='bar'))")
plt.ylabel("อุณหภูมิเฉลี่ย (°C)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
data["temperature_c"].plot(kind="hist", bins=30, figsize=(10, 5), color="steelblue")
plt.title("การกระจายตัวของอุณหภูมิ (วาดด้วย pandas.plot(kind='hist'))")
plt.xlabel("อุณหภูมิ (°C)")
plt.tight_layout()
plt.show()

### 7.2 `pandas.plot()` กับหลายคอลัมน์พร้อมกัน (สร้าง Legend อัตโนมัติ)


In [ ]:
monthly_avg = data.groupby(data["date"].dt.month)[["temperature_c", "humidity_pct"]].mean()

monthly_avg.plot(figsize=(10, 5), marker="o")
plt.title("ค่าเฉลี่ยอุณหภูมิและความชื้นรายเดือน")
plt.xlabel("เดือน")
plt.ylabel("ค่าเฉลี่ย")
plt.legend(["อุณหภูมิ (°C)", "ความชื้น (%)"])
plt.tight_layout()
plt.show()

### 7.3 เปรียบเทียบ matplotlib ตรงๆ vs `pandas.plot()`

| | matplotlib ตรงๆ | `pandas.plot()` |
|---|---|---|
| ความยาวโค้ด | ยาวกว่า ต้องระบุ x, y แยก | สั้นกว่า มาก สำหรับข้อมูลใน DataFrame |
| ความยืดหยุ่น | ยืดหยุ่นที่สุด ปรับแต่งได้ทุกจุด | ยืดหยุ่นน้อยกว่าเล็กน้อย แต่ปรับแต่งต่อด้วย `plt.*` ได้ |
| เหมาะกับ | กราฟที่ต้องการควบคุมรายละเอียดสูง, กราฟซับซ้อน | การสำรวจข้อมูลเร็วๆ (Exploratory Data Analysis) |


---
## 8. seaborn — กราฟสถิติที่สวยและใช้ง่ายขึ้น

**seaborn** สร้างอยู่บน matplotlib แต่ออกแบบมาเฉพาะสำหรับ**กราฟเชิงสถิติ** — โค้ดสั้นกว่า สวยกว่า (มี theme ให้ default) และทำงานกับ DataFrame ได้ลื่นไหลกว่า (ส่ง `data=` และชื่อคอลัมน์ตรงๆ)

### 8.1 Boxplot — ทวนจาก Data Cleaning notebook

ใช้เปรียบเทียบการกระจายตัวระหว่างกลุ่ม และตรวจหา outliers (ตามที่เรียนมาแล้ว)


In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=data, x="region", y="temperature_c")
plt.title("การกระจายตัวของอุณหภูมิ แยกตาม Region")
plt.xlabel("ภูมิภาค")
plt.ylabel("อุณหภูมิ (°C)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 8.2 Boxplot พร้อมแยกสีตามกลุ่มย่อย (`hue`)

`hue` ใส่มิติที่ 3 เข้าไปในกราฟ โดยใช้สีแยกกลุ่มย่อย — มีประโยชน์มากเวลาต้องการเปรียบเทียบ 2 มิติพร้อมกัน


In [ ]:
data["is_weekend"] = data["date"].dt.dayofweek >= 5

plt.figure(figsize=(12, 6))
sns.boxplot(data=data, x="region", y="temperature_c", hue="is_weekend")
plt.title("อุณหภูมิแยกตาม Region และวันธรรมดา/วันหยุด")
plt.xlabel("ภูมิภาค")
plt.ylabel("อุณหภูมิ (°C)")
plt.xticks(rotation=20)
plt.legend(title="วันหยุดสุดสัปดาห์")
plt.tight_layout()
plt.show()

---
## 9. Heatmap Correlation — ความสัมพันธ์ระหว่างตัวแปร

**Correlation (ค่าสหสัมพันธ์)** บอกว่าตัวแปร 2 ตัวมีความสัมพันธ์กันมากแค่ไหน มีค่าระหว่าง **-1 ถึง 1**:
- **ใกล้ +1** → สัมพันธ์ไปทางเดียวกัน (ตัวหนึ่งเพิ่ม อีกตัวก็เพิ่มตาม)
- **ใกล้ -1** → สัมพันธ์ตรงข้ามกัน (ตัวหนึ่งเพิ่ม อีกตัวลด)
- **ใกล้ 0** → ไม่มีความสัมพันธ์เชิงเส้นชัดเจน

`.corr()` ของ pandas คำนวณ correlation ระหว่างทุกคู่คอลัมน์ตัวเลข แล้ว `sns.heatmap()` ทำให้เห็นภาพรวมทั้งหมดในตารางเดียว


In [ ]:
numeric_cols = ["temperature_c", "humidity_pct", "rainfall_mm", "wind_speed_kmh", "pressure_hpa"]
correlation_matrix = data[numeric_cols].corr()
print(correlation_matrix.round(2))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation Heatmap ระหว่างตัวแปรสภาพอากาศ")
plt.tight_layout()
plt.show()

**อ่าน Heatmap อย่างไร:**
- `annot=True` → แสดงตัวเลขค่า correlation ในแต่ละช่อง
- สีแดง (ใกล้ +1) → สัมพันธ์ไปทางเดียวกัน, สีน้ำเงิน (ใกล้ -1) → สัมพันธ์ตรงข้ามกัน
- แนวทแยงมุม (จากซ้ายบนไปขวาล่าง) จะเป็น 1.0 เสมอ (ตัวแปรสัมพันธ์กับตัวมันเองสมบูรณ์แบบ)

**🔍 สังเกต:** ลองดูว่า `temperature_c` กับ `humidity_pct` มีค่า correlation เท่าไหร่ และมีทิศทางไหน (บวก/ลบ) — ในข้อมูลภูมิอากาศจริง อุณหภูมิกับความชื้นมักมีความสัมพันธ์กันในระดับหนึ่งแต่ไม่สมบูรณ์แบบ

**⚠️ ข้อควรระวังสำคัญ:** Correlation บอกแค่ความสัมพันธ์เชิงเส้น **ไม่ได้บอกว่าตัวแปรหนึ่ง "เป็นเหตุ" ของอีกตัวแปร** (correlation ≠ causation) — ต้องระมัดระวังการตีความเสมอ


---
## 10. Pairplot — ดูความสัมพันธ์ทุกคู่ตัวแปรพร้อมกัน

`sns.pairplot()` สร้าง scatter plot ของ**ทุกคู่ตัวแปร**ที่เลือก พร้อมฮิสโตแกรมในแนวทแยง — เป็นวิธีสำรวจข้อมูลเบื้องต้น (EDA) ที่ครอบคลุมและรวดเร็วที่สุด


In [ ]:
# ใช้ sample เพื่อไม่ให้ภาพแน่นเกินไปและรันเร็วขึ้น (pairplot ช้าถ้าข้อมูลมีจำนวนมาก)
sample_data = data[numeric_cols].sample(500, random_state=42)

sns.pairplot(sample_data)
plt.suptitle("Pairplot ของตัวแปรสภาพอากาศทั้งหมด", y=1.02)
plt.show()

**อ่าน Pairplot อย่างไร:**
- **แนวทแยงมุม** → ฮิสโตแกรมของตัวแปรนั้นๆ (การกระจายตัวของตัวมันเอง)
- **ช่องอื่นๆ** → scatter plot ระหว่างตัวแปรในแถวกับคอลัมน์นั้น (เห็นความสัมพันธ์แบบเดียวกับ heatmap แต่เป็นภาพจุดจริง ไม่ใช่แค่ตัวเลขสรุป)

### 10.1 Pairplot พร้อมแยกสีตามกลุ่ม (`hue`)


In [ ]:
sample_with_region = data[numeric_cols + ["region"]].sample(500, random_state=42)

sns.pairplot(sample_with_region, hue="region", plot_kws={"alpha": 0.5})
plt.suptitle("Pairplot แยกสีตาม Region", y=1.02)
plt.show()

**🔍 สังเกต:** การแยกสีตาม `region` ช่วยให้เห็นว่าความสัมพันธ์ระหว่างตัวแปรอาจ**แตกต่างกันในแต่ละกลุ่ม** — เป็นเทคนิคที่มีประโยชน์มากในขั้นตอนสำรวจข้อมูลก่อนตัดสินใจว่าจะสร้างโมเดลแยกตามกลุ่มหรือไม่

> ⚠️ **ข้อควรระวัง:** `pairplot()` คำนวณกราฟจำนวนมาก (N² กราฟ เมื่อ N คือจำนวนคอลัมน์) จึง**ช้าและกินทรัพยากรมาก**ถ้าข้อมูลมีหลายคอลัมน์หรือหลายแถว ควร `.sample()` ข้อมูลก่อนเสมอเมื่อข้อมูลมีขนาดใหญ่ (เหมือนตัวอย่างที่ทำไว้ข้างบน)


---
# ส่วนที่ 3: Visualization for Insight

จนถึงตอนนี้รู้จักกราฟหลายชนิดแล้ว — หัวข้อนี้จะเน้น **"การตัดสินใจ"** ว่าควรใช้กราฟแบบไหนกับข้อมูลแบบไหน และจะทำให้กราฟ "สื่อ insight" ได้จริง ไม่ใช่แค่ "วาดได้"

## 11. เลือก Chart Type ให้เหมาะกับข้อมูล

| คำถามที่อยากตอบ | Chart Type ที่เหมาะ | ตัวอย่าง |
|---|---|---|
| แนวโน้มตามเวลาเปลี่ยนแปลงอย่างไร? | **Line Chart** | อุณหภูมิรายวันของเมืองหนึ่งตลอด 90 วัน |
| ตัวแปร 2 ตัวสัมพันธ์กันไหม? | **Scatter Plot** | ความชื้น vs ปริมาณฝน |
| เปรียบเทียบค่าระหว่างกลุ่ม (หมวดหมู่)? | **Bar Chart** | อุณหภูมิเฉลี่ยของแต่ละภูมิภาค |
| ข้อมูลตัวเลขกระจายตัวอย่างไร? | **Histogram** | การกระจายตัวของอุณหภูมิทั้งหมด |
| เปรียบเทียบการกระจายตัวระหว่างกลุ่ม + หา outliers? | **Boxplot** | อุณหภูมิแยกตามภูมิภาค |
| ความสัมพันธ์ระหว่างตัวแปรหลายคู่พร้อมกัน? | **Heatmap / Pairplot** | correlation ระหว่างตัวแปรสภาพอากาศทั้งหมด |
| สัดส่วนของส่วนประกอบทั้งหมด? | **Pie Chart** (ใช้น้อยกว่าที่คิด — ดูหัวข้อถัดไป) | สัดส่วนจำนวนวันที่ฝนตก/ไม่ตก |

### 11.1 กรณีศึกษา: เลือกผิดประเภทแล้วสื่อความหมายผิด

ลองดูตัวอย่างที่ใช้กราฟผิดประเภทกับคำถามที่ต้องการตอบ


In [ ]:
# ❌ ตัวอย่างไม่ดี: ใช้ Line Chart กับข้อมูลที่ไม่มีลำดับเวลาที่ต่อเนื่องกัน (เมืองเรียงตามตัวอักษร ไม่ใช่ตามเวลา)
avg_by_city = data.groupby("city")["temperature_c"].mean().sort_index()   # เรียงตามชื่อ A-Z

plt.figure(figsize=(12, 5))
plt.plot(avg_by_city.index, avg_by_city.values, marker="o")
plt.title("อุณหภูมิเฉลี่ยแต่ละเมือง (ใช้ Line Chart -- ตัวอย่างที่ไม่เหมาะ)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("ปัญหา: เส้นเชื่อมระหว่างจุดบอกเป็นนัยว่ามี 'ความต่อเนื่อง' ระหว่างเมือง "
      "(เช่น เหมือนบอกว่า Auckland 'ไหลต่อ' ไปเป็น Bangkok) ซึ่งไม่มีความหมายอะไรเลย "
      "เพราะลำดับเมืองเรียงตามตัวอักษร ไม่ใช่ปริมาณที่ต่อเนื่องกันจริง")

In [ ]:
# ✅ ตัวอย่างที่ดีกว่า: ใช้ Bar Chart กับข้อมูลเชิงหมวดหมู่ (ไม่มีเส้นเชื่อมที่บอกความต่อเนื่องผิดๆ)
avg_by_city_sorted = data.groupby("city")["temperature_c"].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
plt.bar(avg_by_city_sorted.index, avg_by_city_sorted.values, color="steelblue")
plt.title("อุณหภูมิเฉลี่ยแต่ละเมือง เรียงจากร้อนสุดไปเย็นสุด")
plt.ylabel("อุณหภูมิเฉลี่ย (°C)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("ดีขึ้น: Bar Chart ไม่บอกเป็นนัยถึงความต่อเนื่อง และการเรียงจากมากไปน้อย "
      "ช่วยให้เห็นการเปรียบเทียบระหว่างเมืองได้ทันทีโดยไม่ต้องไล่หาด้วยตา")

### 11.2 กรณีศึกษา: Pie Chart ใช้ยากกว่าที่คิด

Pie Chart เป็นที่นิยมแต่จริงๆ**มนุษย์เปรียบเทียบมุม/พื้นที่ได้ไม่แม่นยำเท่าเปรียบเทียบความสูงของแท่ง** — ส่วนใหญ่ Bar Chart สื่อสารได้ดีกว่า โดยเฉพาะเมื่อมีหลายหมวดหมู่


In [ ]:
rain_level_counts = pd.cut(
    data["rainfall_mm"], bins=[-0.1, 0, 5, 15, 100],
    labels=["ไม่มีฝน", "ฝนน้อย", "ฝนปานกลาง", "ฝนมาก"]
).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(rain_level_counts.values, labels=rain_level_counts.index, autopct="%1.1f%%")
axes[0].set_title("Pie Chart: สัดส่วนระดับฝนตก")

axes[1].bar(rain_level_counts.index, rain_level_counts.values, color="steelblue")
axes[1].set_title("Bar Chart: จำนวนวันแต่ละระดับฝนตก")
axes[1].set_ylabel("จำนวนวัน")

plt.tight_layout()
plt.show()

print("ลองเทียบ: Bar Chart (ขวา) อ่านค่าและเปรียบเทียบระหว่างกลุ่มได้แม่นยำกว่า Pie Chart (ซ้าย) "
      "อย่างเห็นได้ชัด โดยเฉพาะเมื่อสัดส่วนของบางกลุ่มใกล้เคียงกัน")

---
## 12. ตั้งชื่อกราฟให้สื่อความหมาย

ชื่อกราฟที่ดีไม่ใช่แค่ "บอกว่ากราฟนี้คือข้อมูลอะไร" แต่ควร **"บอก insight ที่อยากให้คนดูเห็น"** ด้วย — เปรียบเทียบ 2 แบบ

| ชื่อกราฟแบบทั่วไป (Descriptive) | ชื่อกราฟแบบสื่อ Insight (Insight-driven) |
|---|---|
| "อุณหภูมิเฉลี่ยแยกตามภูมิภาค" | "เอเชียและแอฟริการ้อนกว่ายุโรปเฉลี่ย 15 องศา" |
| "ความสัมพันธ์ระหว่างความชื้นและฝน" | "ความชื้นสูงกว่า 80% มักมาพร้อมฝนตกหนัก" |
| "อุณหภูมิรายวันที่กรุงเทพฯ" | "กรุงเทพฯร้อนขึ้นต่อเนื่องตลอดเดือนมีนาคม" |

**หลักการ:** ชื่อแบบ Descriptive บอกแค่ "ตัวแปรอะไรอยู่ในกราฟ" ส่วนชื่อแบบ Insight-driven บอก **"สิ่งที่ค้นพบ"** จากกราฟนั้น — ผู้อ่านที่เร่งรีบอาจดูแค่ชื่อกราฟก็ได้ข้อมูลสำคัญไปแล้ว โดยไม่ต้องตีความกราฟเอง

### 12.1 ตัวอย่าง: เปลี่ยนชื่อกราฟจาก Descriptive เป็น Insight-driven


In [ ]:
avg_temp_by_region = data.groupby("region")["temperature_c"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# กราฟซ้าย: ชื่อแบบทั่วไป
axes[0].bar(avg_temp_by_region.index, avg_temp_by_region.values, color="lightgray")
axes[0].set_title("อุณหภูมิเฉลี่ยแยกตามภูมิภาค", fontsize=12)
axes[0].set_ylabel("อุณหภูมิเฉลี่ย (°C)")
axes[0].tick_params(axis="x", rotation=20)

# กราฟขวา: ชื่อแบบสื่อ insight + เน้นสีที่ประเด็นสำคัญ
colors = ["firebrick" if r in ["Asia", "Africa"] else "lightgray" for r in avg_temp_by_region.index]
diff = avg_temp_by_region["Asia"] - avg_temp_by_region["Europe"]
axes[1].bar(avg_temp_by_region.index, avg_temp_by_region.values, color=colors)
axes[1].set_title(f"เอเชียร้อนกว่ายุโรปเฉลี่ย {diff:.1f}°C", fontsize=12, fontweight="bold")
axes[1].set_ylabel("อุณหภูมิเฉลี่ย (°C)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

**🔍 สังเกต:** กราฟขวาทำ 2 อย่างที่กราฟซ้ายไม่ได้ทำ:
1. **ชื่อกราฟบอก insight ตรงๆ** (ไม่ต้องตีความเอง)
2. **ใช้สีเน้น (highlight)** เฉพาะแท่งที่เกี่ยวข้องกับ insight นั้น ส่วนแท่งอื่นใช้สีเรียบเพื่อไม่ให้แย่งความสนใจ

### 12.2 หลักการตั้งชื่อกราฟที่ดี

- **เจาะจง** — ใส่ตัวเลขจริงถ้าเป็นไปได้ (เช่น "ร้อนกว่า 15°C" ดีกว่า "ร้อนกว่ามาก")
- **สั้นกระชับ** — 1 บรรทัด อ่านจบในไม่กี่วินาที
- **ตรงประเด็น** — ตอบคำถามที่กราฟนี้ถูกสร้างมาเพื่อตอบ ไม่ใช่บอกแค่ "นี่คือกราฟอะไร"
- **ไม่กล่าวเกินจริง** — insight ต้องสอดคล้องกับข้อมูลจริง ไม่ใช่บีบให้ข้อมูลสนับสนุนสิ่งที่อยากพูด


---
## 13. Storytelling เบื้องต้นด้วยข้อมูล

**Data Storytelling** คือการนำเสนอข้อมูลเป็น "เรื่องราว" ที่มีลำดับ ชัดเจน และโน้มน้าวใจ — ไม่ใช่แค่โชว์กราฟสวยๆ แต่นำกราฟหลายอันมาเรียงเป็นลำดับเพื่อ "เล่าเรื่อง" ให้ผู้ฟังเข้าใจประเด็นทีละขั้น

### โครงสร้าง Storytelling เบื้องต้น (3 ขั้น)

1. **Context (ตั้งคำถาม)** — เริ่มด้วยคำถามหรือบริบทที่ผู้ฟังสนใจ
2. **Evidence (แสดงหลักฐาน)** — ใช้กราฟค่อยๆ คลี่คลายคำถามนั้นทีละขั้น
3. **Conclusion (สรุปและเสนอแนะ)** — สรุป insight และ (ถ้าเหมาะสม) เสนอแนะสิ่งที่ควรทำต่อ

### ตัวอย่าง Storytelling แบบสมบูรณ์: "เมืองไหนในเอเชียที่ฝนตกบ่อยที่สุด และเกี่ยวข้องกับความชื้นอย่างไร?"

**ขั้นที่ 1 — Context:** ตั้งคำถามให้ชัดก่อนเริ่มดูข้อมูล


In [ ]:
asian_data = data[data["region"] == "Asia"]
print(f"กำลังวิเคราะห์ข้อมูล {len(asian_data)} แถว จาก {asian_data['city'].nunique()} เมืองในเอเชีย")
print("คำถาม: เมืองไหนฝนตกบ่อยที่สุด และความชื้นมีความสัมพันธ์กับฝนแค่ไหน?")

**ขั้นที่ 2 — Evidence (กราฟที่ 1):** เริ่มจากภาพรวมกว้างๆ


In [ ]:
rainy_days_by_city = (asian_data[asian_data["rainfall_mm"] > 5]
                       .groupby("city").size()
                       .sort_values(ascending=False))

plt.figure(figsize=(10, 5))
colors = ["firebrick" if i == 0 else "lightgray" for i in range(len(rainy_days_by_city))]
plt.bar(rainy_days_by_city.index, rainy_days_by_city.values, color=colors)
plt.title(f"{rainy_days_by_city.index[0]} มีวันฝนตกมากที่สุดในเอเชีย ({rainy_days_by_city.iloc[0]} วัน จาก 90 วัน)")
plt.ylabel("จำนวนวันที่ฝนตก (>5mm)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**ขั้นที่ 2 — Evidence (กราฟที่ 2):** เจาะลึกเมืองที่น่าสนใจที่สุดจากกราฟแรก


In [ ]:
top_city = rainy_days_by_city.index[0]
top_city_data = asian_data[asian_data["city"] == top_city]

plt.figure(figsize=(8, 6))
plt.scatter(top_city_data["humidity_pct"], top_city_data["rainfall_mm"], alpha=0.5, color="steelblue")
correlation = top_city_data["humidity_pct"].corr(top_city_data["rainfall_mm"])
plt.title(f"{top_city}: ความชื้นและฝนมีความสัมพันธ์กัน (correlation = {correlation:.2f})")
plt.xlabel("ความชื้น (%)")
plt.ylabel("ปริมาณฝน (mm)")
plt.tight_layout()
plt.show()

**ขั้นที่ 3 — Conclusion:** สรุปสิ่งที่ค้นพบจากทั้ง 2 กราฟ


In [ ]:
print(f"สรุปผลการวิเคราะห์:")
print(f"1. {top_city} เป็นเมืองที่ฝนตกบ่อยที่สุดในเอเชียจากข้อมูลนี้ ({rainy_days_by_city.iloc[0]} วัน จาก 90 วัน)")
print(f"2. ความชื้นและปริมาณฝนใน {top_city} มี correlation = {correlation:.2f} "
      f"({'สัมพันธ์ไปทางเดียวกันระดับปานกลางถึงสูง' if correlation > 0.3 else 'สัมพันธ์กันค่อนข้างน้อย'})")
print(f"3. ข้อเสนอแนะ: หากต้องวางแผนกิจกรรมกลางแจ้งใน {top_city} ควรติดตามค่าความชื้นเป็นตัวบ่งชี้เบื้องต้นของโอกาสฝนตก")

### 13.1 ข้อคิดสำคัญเกี่ยวกับ Storytelling

- **เรียงกราฟจากกว้างไปแคบ (Overview → Detail)** ช่วยให้ผู้ฟังตามเรื่องราวได้ง่ายกว่าโชว์กราฟละเอียดทันที
- **ทุกกราฟควรมีหน้าที่ในเรื่องราว** — ถ้ากราฟไม่ได้ช่วยตอบคำถามหรือนำไปสู่ข้อสรุป ควรตัดออก (Less is more)
- **Conclusion ต้องเชื่อมโยงกับ Context ที่ตั้งไว้ตอนแรก** — กลับไปตอบคำถามเริ่มต้นเสมอ
- **ความซื่อตรงต่อข้อมูลสำคัญที่สุด** — Storytelling ไม่ใช่การ "เลือกเฉพาะข้อมูลที่สนับสนุนเรื่องที่อยากเล่า" แต่คือการนำเสนอสิ่งที่ข้อมูลบอกจริงๆให้เข้าใจง่ายที่สุด


---
## 🧪 แบบฝึกหัดท้ายคาบ

> ลองทำเองในเซลล์ที่เตรียมไว้ด้านล่างแต่ละข้อ ไม่มีเฉลยให้ — ถ้าไม่แน่ใจให้ลองรันดูผลลัพธ์ หรือถามอาจารย์/เพื่อนได้เลย ใช้ตัวแปร `data` ที่ merge และทำความสะอาดแล้วตลอดทั้งไฟล์นี้

### ข้อ 1: matplotlib พื้นฐาน
สร้าง 4 กราฟแยกกัน (ใส่ title, xlabel, ylabel ให้ครบทุกกราฟ):
1. Line chart: อุณหภูมิรายวันของเมือง `"Sydney"`
2. Scatter plot: `pressure_hpa` กับ `wind_speed_kmh`
3. Bar chart: ปริมาณฝนเฉลี่ยแยกตาม `region`
4. Histogram: การกระจายตัวของ `wind_speed_kmh`


In [ ]:
# เขียนคำตอบข้อ 1 ที่นี่


### ข้อ 2: Subplot
สร้าง subplot ขนาด 2x2 ที่รวม 4 กราฟจากข้อ 1 ไว้ในภาพเดียว พร้อม `plt.suptitle()` ตั้งชื่อรวมให้ทั้งภาพ


In [ ]:
# เขียนคำตอบข้อ 2 ที่นี่


### ข้อ 3: seaborn
1. สร้าง boxplot เปรียบเทียบ `rainfall_mm` แยกตาม `region` พร้อม `hue` แยกตาม `is_weekend`
2. สร้าง correlation heatmap ระหว่าง `temperature_c`, `humidity_pct`, `rainfall_mm`, `population_millions`, `avg_gdp_growth_pct` (คอลัมน์ที่ได้มาจากการ merge ตอนต้นไฟล์)
3. ตอบในคอมเมนต์: มีคู่ตัวแปรไหนที่ correlation สูงเกินคาด หรือต่ำกว่าที่คาดไว้ไหม?


In [ ]:
# เขียนคำตอบข้อ 3 ที่นี่


### ข้อ 4: เลือก Chart Type ให้เหมาะสม
สำหรับ 3 สถานการณ์นี้ จงเลือกและสร้างกราฟที่เหมาะสมที่สุด (พร้อมเหตุผลในคอมเมนต์ว่าทำไมเลือกแบบนี้):
1. อยากดูว่า `population_millions` ของเมืองมีผลต่อ `temperature_c` เฉลี่ยหรือไม่
2. อยากเปรียบเทียบจำนวนเมืองในแต่ละ `region`
3. อยากดูว่าอุณหภูมิของเมือง `"Moscow"` เปลี่ยนแปลงอย่างไรในช่วง 90 วัน


In [ ]:
# เขียนคำตอบข้อ 4 ที่นี่


### ข้อ 5: ตั้งชื่อกราฟให้สื่อความหมาย
1. สร้างกราฟ bar chart เปรียบเทียบ `population_millions` เฉลี่ยของแต่ละ `region` (ใช้ `.drop_duplicates(subset="city")` ก่อน group เพื่อไม่นับประชากรซ้ำหลายครั้งต่อเมือง)
2. ตั้งชื่อกราฟ 2 แบบ: แบบ Descriptive ธรรมดา 1 ชื่อ และแบบ Insight-driven (มีตัวเลขประกอบ) อีก 1 ชื่อ
3. ใช้สีเน้น (highlight) เฉพาะ region ที่มีประชากรเฉลี่ยสูงสุด เหมือนตัวอย่างที่เรียนมา


In [ ]:
# เขียนคำตอบข้อ 5 ที่นี่


### ข้อ 6: ท้าทายเพิ่มเติม (Challenge) — Storytelling แบบสมบูรณ์
จงเลือกคำถามที่น่าสนใจของตัวเอง 1 คำถามเกี่ยวกับข้อมูลนี้ (เช่น "เมืองไหนมีความเสี่ยงด้านสภาพอากาศรุนแรงมากที่สุด" หรือคำถามอื่นที่สนใจ) แล้วสร้าง Storytelling แบบ 3 ขั้นตอน (Context → Evidence → Conclusion) ให้ครบ:
1. เขียน Context อธิบายคำถามที่จะตอบ (เป็น markdown cell หรือ print ก็ได้)
2. สร้างกราฟอย่างน้อย 2 กราฟที่ค่อยๆ คลี่คลายคำถามนั้น (เรียงจากภาพรวมไปสู่รายละเอียด)
3. เขียน Conclusion สรุปสิ่งที่ค้นพบและข้อเสนอแนะ (ถ้ามี)


In [ ]:
# เขียนคำตอบข้อ 6 ที่นี่


---
## 🔗 เชื่อม Colab กับ GitHub

เก็บ Notebook นี้ขึ้น GitHub ต่อจากไฟล์ก่อนหน้า

### วิธีที่ 1: เซฟจาก Colab ขึ้น GitHub ตรงๆ (สำหรับ Notebook)

1. ใน Colab ไปที่เมนู **File → Save a copy in GitHub**
2. เลือก repository เดิมที่ใช้เก็บ Notebook คาบก่อนๆ (เช่น `SC612104-coursework`)
3. ตั้งชื่อไฟล์และ commit message (เช่น `"เพิ่ม notebook: data visualization (matplotlib/seaborn/storytelling)"`) แล้วกด **OK**

### วิธีที่ 2: ใช้ Git ผ่าน Terminal

```bash
git clone https://github.com/<username>/<repo-name>.git
cd <repo-name>
git add notebook.ipynb
git commit -m "เพิ่ม notebook: data visualization"
git push origin main
```

> 💡 **Tip:** ทักษะการเลือกกราฟให้เหมาะสมและตั้งชื่อให้สื่อความหมาย มักสำคัญกว่าทักษะการเขียนโค้ดสร้างกราฟเสียอีก — นักวิเคราะห์ข้อมูลที่ดีคือคนที่รู้ว่า "จะแสดงอะไร" และ "ทำไมต้องแสดงสิ่งนั้น" ไม่ใช่แค่รู้คำสั่งสร้างกราฟทุกแบบ ลองฝึกตั้งคำถามกับข้อมูลก่อนเปิดคอมพิวเตอร์สร้างกราฟทุกครั้ง จะช่วยให้กราฟที่ได้มีจุดมุ่งหมายชัดเจนกว่าการลองสร้างกราฟทุกแบบแล้วดูว่าอันไหนสวย
